In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

figure_dir = "figures/revision/figure-1"
setup_plotting(figure_dir, display_dpi=300, save_dpi=300)

import scanpy as sc

from spatial_tcr.clonal_expansion import annotate_singlets, get_avbv_expression
from spatial_tcr.tcr import get_tcr_genes

In [ ]:
data_dir = "data/xenium/processed"
path = f"{data_dir}/06.2-kidney_tcr_filtered.h5ad"
adata = sc.read_h5ad(path)
adata

In [ ]:
adata.obs["cell_type_l1.1"] = adata.obs["cell_type_l2"].astype(str)
adata.obs["cell_type_l1.1"] = adata.obs["cell_type_l1.1"].replace({"Tregs": "CD4+"})
adata.obs["cell_type_l1.1"].value_counts()

In [ ]:
av_genes, bv_genes, dv_genes, gv_genes, tv_genes = get_tcr_genes(adata)
annotate_singlets(adata, genes=av_genes, out_key="av_clone", allow_multiplets=True)
annotate_singlets(adata, genes=bv_genes, out_key="bv_clone", allow_multiplets=True)

In [ ]:
adata.obs["bv_clone"].value_counts(dropna=True).sum()

In [ ]:
adata.obs["bv_clone"].value_counts(dropna=True)

In [ ]:
adata.obs["avbv_clone"] = (
    adata.obs["av_clone"].astype(str) + "+" + adata.obs["bv_clone"].astype(str)
)
# remove None values
adata.obs.loc[adata.obs["av_clone"].isna(), "avbv_clone"] = None
adata.obs.loc[adata.obs["bv_clone"].isna(), "avbv_clone"] = None
adata.obs["avbv_clone"].value_counts(dropna=True).sum()

In [ ]:
df_avbv = get_avbv_expression(adata, layer="counts")
df_avbv.head()

In [ ]:
adata.write_h5ad(f"{data_dir}/07.1-kidney_tcr_clones.h5ad")